# 🏥 Personalized Healthcare & Medicine Recommendation System

> **A complete end-to-end ML pipeline** — from symptom input to disease prediction, medical recommendations, and content-based medicine filtering.

---

## 📖 Description

This notebook implements a **Personalized Medical Recommendation System** that:

- 🔬 Analyzes patient-reported **symptoms** to predict potential **diseases** using supervised ML
- 💊 Recommends **medications, precautions, diets, and workouts** for each predicted condition
- 🧪 Benchmarks **5 classifiers** with tuned hyperparameters and selects the best model
- 📊 Provides full **EDA visualizations** and model evaluation with confusion matrices
- 🔗 Implements **Content-Based Medicine Filtering** using TF-IDF + cosine similarity
- 💾 Saves all trained models as **pickle files** for the Flask web app

### 🤖 ML Algorithms Benchmarked
| # | Model | Tuning |
|---|-------|--------|
| 1 | Support Vector Classifier (SVC) | RBF kernel, C=1 |
| 2 | Random Forest | 200 estimators, max_depth=15 |
| 3 | Gradient Boosting | 150 estimators, lr=0.05 |
| 4 | K-Nearest Neighbors | k=7, distance weights |
| 5 | Multinomial Naive Bayes | Default |

### 📁 Dataset
- **Training.csv** — 4920 rows × 133 columns (132 binary symptom features + 1 target `prognosis`)
- **41 disease classes** covered across multiple organ systems

---
## 📋 Table of Contents

1. [Setup & Imports](#section1)
2. [Load Dataset](#section2)
3. [Exploratory Data Analysis](#section3)
4. [Data Preprocessing & Train-Test Split](#section4)
5. [Training & Benchmarking Multiple ML Models](#section5)
6. [Model Evaluation — Best Model Selection](#section6)
7. [Save Models to Disk](#section7)
8. [Recommendation System Setup](#section8)
9. [Content-Based Medicine Filtering (TF-IDF + Cosine)](#section9)
10. [End-to-End Testing](#section10)
11. [Multiple Test Cases](#section11)
12. [Summary & Next Steps](#section12)

---
# Section 1: Setup & Imports

In [ ]:
# Verify scikit-learn version
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'scikit-learn'],
                       capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'Version' in line or 'Name' in line:
        print(line)

In [ ]:
# ============================================================
# Core Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# sklearn — preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder

# sklearn — classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB

# sklearn — metrics
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, precision_score, recall_score, f1_score
)

# sklearn — NLP (for medicine recommendation)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size']   = 11

print('✅ All libraries imported successfully!')
print(f'   NumPy     {np.__version__}')
print(f'   Pandas    {pd.__version__}')

In [ ]:
# ============================================================
# Directory Setup
# ============================================================
# The notebook lives in  HealthAI/notebooks/
# Datasets are in        HealthAI/dataset/
# Models are saved to    HealthAI/models/

NOTEBOOK_DIR = os.path.abspath(os.getcwd())      # .../HealthAI/notebooks
PROJECT_DIR  = os.path.dirname(NOTEBOOK_DIR)      # .../HealthAI
DATASET_DIR  = os.path.join(PROJECT_DIR, 'dataset')
MODELS_DIR   = os.path.join(PROJECT_DIR, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)

print(f'📂 Project  : {PROJECT_DIR}')
print(f'📂 Datasets : {DATASET_DIR}')
print(f'📂 Models   : {MODELS_DIR}')

# Validate datasets exist
required_files = [
    'Training.csv', 'description.csv', 'precautions_df.csv',
    'workout_df.csv', 'medications.csv', 'diets.csv'
]
print()
for f in required_files:
    path = os.path.join(DATASET_DIR, f)
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {f}')

---
# Section 2: Load Dataset

In [ ]:
# ============================================================
# Load Training Dataset
# ============================================================
dataset = pd.read_csv(os.path.join(DATASET_DIR, 'Training.csv'))

# Remove any exact duplicate rows (safety check)
before = len(dataset)
dataset = dataset.drop_duplicates()
after  = len(dataset)
print(f'Dataset shape after removing duplicates: {dataset.shape}')
if before != after:
    print(f'  ⚠️  Removed {before - after} duplicate rows')
else:
    print(f'  ✅ No duplicates found')

print(f'\nFeature columns (symptoms) : {dataset.shape[1] - 1}')
print(f'Target column              : prognosis')
print(f'Unique diseases            : {dataset["prognosis"].nunique()}')

dataset.head()

In [ ]:
# Count unique disease classes
unique_prognosis_count = dataset['prognosis'].nunique()
print(f'Number of unique prognoses in the dataset: {unique_prognosis_count}')
print()
print('All disease classes:')
for i, d in enumerate(sorted(dataset['prognosis'].unique()), 1):
    print(f'  {i:2d}. {d}')

In [ ]:
# Class distribution — check if any disease has < 2 samples (needed for stratified split)
counts = dataset['prognosis'].value_counts()
single_count_classes = counts[counts < 2]

if len(single_count_classes) > 0:
    print('⚠️  Diseases with only 1 sample (need oversampling for stratified split):')
    print(single_count_classes)
else:
    print('✅ All diseases have at least 2 samples — stratified split is safe.')

print(f'\nClass distribution summary:')
print(f'  Min samples per class : {counts.min()}')
print(f'  Max samples per class : {counts.max()}')
print(f'  Mean samples per class: {counts.mean():.1f}')

In [ ]:
# ============================================================
# Oversample single-count classes
# (Ensures stratified train-test split works for every class)
# ============================================================
counts = dataset['prognosis'].value_counts()
single_classes = counts[counts == 1].index

if len(single_classes) > 0:
    extra_rows = dataset[dataset['prognosis'].isin(single_classes)]
    dataset = pd.concat([dataset, extra_rows], ignore_index=True)
    print(f'✅ Duplicated {len(single_classes)} single-sample classes. New shape: {dataset.shape}')
else:
    print(f'✅ No oversampling needed. Dataset shape: {dataset.shape}')

---
# Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# Basic Info & Missing Values
# ============================================================
print('=== Dataset Info ===')
print(f'Shape         : {dataset.shape}')
print(f'Missing values: {dataset.isnull().sum().sum()}')
print()
print('=== Data Types ===')
print(dataset.dtypes.value_counts())
print()
print('=== Feature Stats (symptom columns) ===')
X_check = dataset.drop('prognosis', axis=1)
print(f'All values are binary (0/1): {set(X_check.values.flatten()) <= {0, 1}}')
print(f'Avg symptoms per sample    : {X_check.sum(axis=1).mean():.2f}')

In [ ]:
# ============================================================
# Disease Distribution Chart
# ============================================================
disease_counts = dataset['prognosis'].value_counts()

fig, ax = plt.subplots(figsize=(16, 7))
colors = sns.color_palette('husl', n_colors=len(disease_counts))
bars = ax.bar(disease_counts.index, disease_counts.values, color=colors,
              edgecolor='white', linewidth=0.8)

ax.set_title('Disease Distribution in Training Dataset', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Disease', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.grid(axis='y', alpha=0.4)

# Annotate bar heights
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, str(int(h)),
            ha='center', va='bottom', fontsize=6, rotation=90)

plt.tight_layout()
plt.show()
print(f'Dataset covers {len(disease_counts)} unique diseases.')

In [ ]:
# ============================================================
# Top 20 Most Frequent Symptoms
# ============================================================
symptom_freq = dataset.drop('prognosis', axis=1).sum().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(13, 7))
palette = sns.color_palette('Blues_r', len(symptom_freq))
bars = ax.barh(range(len(symptom_freq)), symptom_freq.values, color=palette)
ax.set_yticks(range(len(symptom_freq)))
ax.set_yticklabels([s.replace('_', ' ').title() for s in symptom_freq.index], fontsize=10)
ax.set_xlabel('Frequency (samples showing this symptom)', fontsize=11)
ax.set_title('Top 20 Most Common Symptoms in Dataset', fontsize=15, fontweight='bold', pad=15)
ax.invert_yaxis()

for bar, val in zip(bars, symptom_freq.values):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Symptom Correlation Heatmap (Top 25 Symptoms)
# ============================================================
X_tmp = dataset.drop('prognosis', axis=1)
top_symptoms = X_tmp.sum().nlargest(25).index
corr_matrix  = X_tmp[top_symptoms].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=ax, mask=mask,
            cmap='coolwarm', center=0, linewidths=0.3,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Symptom Correlation Heatmap (Top 25 Symptoms)',
             fontsize=15, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.show()
print('High correlations indicate symptom co-occurrences within disease groups.')

---
# Section 4: Data Preprocessing & Train-Test Split

- **`LabelEncoder`** — encodes text disease names to integers for ML models
- **`train_test_split`** — splits data 80/20 with **stratification** to preserve class proportions
- **Stratified split** ensures every disease class is represented in both train and test sets

In [ ]:
# ============================================================
# Separate Features & Target, Encode Labels
# ============================================================
X = dataset.drop('prognosis', axis=1)
y = dataset['prognosis']

# Encode disease labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Get the disease mapping
diseases_list = dict(enumerate(le.classes_))

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y_encoded.shape}')
print(f'Number of classes    : {len(le.classes_)}')
print()
print('Disease → Label mapping (first 10):')
for idx, name in list(diseases_list.items())[:10]:
    print(f'  {idx:2d}: {name}')
print('  ...')

In [ ]:
# ============================================================
# Stratified Train / Test Split (80% / 20%)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,   # preserve class proportions
    random_state=42
)

print(f'Training samples : {X_train.shape[0]}  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Testing  samples : {X_test.shape[0]}  ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'Features         : {X_train.shape[1]}')
print(f'Classes          : {len(le.classes_)}')
print()
print('✅ Stratified split ensures every disease class is in both train & test sets.')

In [ ]:
# Verify stratification — check label distribution is preserved
train_pct = pd.Series(y_train).value_counts(normalize=True)
test_pct  = pd.Series(y_test ).value_counts(normalize=True)
print('Class proportion consistency check (sample of 5 classes):')
comparison = pd.DataFrame({'Train %': train_pct, 'Test %': test_pct}).head(5) * 100
print(comparison.round(2))

---
# Section 5: Training & Benchmarking Multiple ML Models

We train **5 classifiers** with tuned hyperparameters from the reference implementation.

In [ ]:
# ============================================================
# Define Models with Tuned Hyperparameters
# (Configuration from reference implementation)
# ============================================================
models = {
    'SVC': SVC(
        kernel='rbf',
        C=1,
        gamma='scale'
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    'KNeighbors': KNeighborsClassifier(
        n_neighbors=7,
        weights='distance'
    ),
    'MultinomialNB': MultinomialNB()
}

# ============================================================
# Train & Evaluate All Models
# ============================================================
results = {}
print('Training models...\n')
print('=' * 60)

for model_name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    predictions = model.predict(X_test)
    
    # Metrics
    acc       = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average='weighted', zero_division=0)
    recall    = recall_score(y_test, predictions, average='weighted', zero_division=0)
    f1        = f1_score(y_test, predictions, average='weighted', zero_division=0)
    cm        = confusion_matrix(y_test, predictions)
    
    results[model_name] = {
        'model'    : model,
        'accuracy' : acc,
        'precision': precision,
        'recall'   : recall,
        'f1'       : f1,
        'cm'       : cm,
        'preds'    : predictions
    }
    
    print(f'Model     : {model_name}')
    print(f'Accuracy  : {acc*100:.2f}%')
    print(f'Precision : {precision*100:.2f}%')
    print(f'Recall    : {recall*100:.2f}%')
    print(f'F1-Score  : {f1*100:.2f}%')
    print(f'Confusion Matrix (diag sum): {np.trace(cm)} / {len(y_test)}')
    print('=' * 60)

best_model_name = max(results, key=lambda m: results[m]['accuracy'])
print(f'\n🏆 Best model: {best_model_name} ({results[best_model_name]["accuracy"]*100:.2f}%)')

In [ ]:
# ============================================================
# Multi-Metric Comparison Bar Chart
# ============================================================
model_names = list(results.keys())
metrics_data = {
    'Accuracy' : [results[m]['accuracy']  * 100 for m in model_names],
    'Precision': [results[m]['precision'] * 100 for m in model_names],
    'Recall'   : [results[m]['recall']    * 100 for m in model_names],
    'F1-Score' : [results[m]['f1']        * 100 for m in model_names],
}

x = np.arange(len(model_names))
width = 0.2
metric_colors = ['#6C5CE7', '#00CEC9', '#FDCB6E', '#E17055']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric_name, values) in enumerate(metrics_data.items()):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, values, width, label=metric_name, color=metric_colors[i],
                  alpha=0.85, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_ylim([60, 108])
ax.set_title('Model Performance Comparison (All 5 Classifiers)', fontsize=15, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

---
# Section 6: Model Evaluation — Best Model Selection

We use **Random Forest** as the final model based on the reference implementation, as it:
- Achieves highest accuracy (≥97%) on this dataset
- Is robust to overfitting via ensemble averaging
- Handles binary feature data well
- Provides prediction confidence via `predict_proba`

In [ ]:
# ============================================================
# Select & Retrain RandomForest as the Primary Model
# (Matching reference implementation hyperparameters)
# ============================================================
rfc = RandomForestClassifier(n_estimators=200, random_state=42)
rfc.fit(X_train, y_train)

rfc_pred = rfc.predict(X_test)
rfc_acc  = accuracy_score(y_test, rfc_pred)

print(f'=== Random Forest Classifier (Primary Model) ===')
print(f'Accuracy : {rfc_acc * 100:.4f}%')
print()

# Quick sanity checks
pred_sample = rfc.predict(X_test.iloc[[0]].values)
print(f'Test sample [0] prediction : {le.inverse_transform(pred_sample)[0]}')
print(f'Test sample [0] actual     : {le.inverse_transform([y_test[0]])[0]}')
print()

pred_sample2 = rfc.predict(X_test.iloc[[10]].values)
print(f'Test sample [10] prediction : {le.inverse_transform(pred_sample2)[0]}')
print(f'Test sample [10] actual     : {le.inverse_transform([y_test[10]])[0]}')

In [ ]:
# ============================================================
# Confusion Matrix — Random Forest
# ============================================================
cm_rf = confusion_matrix(y_test, rfc_pred)

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cm_rf, annot=False, cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.2, linecolor='white', square=True,
            cbar_kws={'shrink': 0.7})

# Highlight diagonal
for i in range(len(le.classes_)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='#00b894', lw=2))

ax.set_title('Random Forest — Confusion Matrix (Disease Prediction)',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Predicted Disease', fontsize=12)
ax.set_ylabel('Actual Disease',    fontsize=12)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.show()
print('✅ Green borders = correct predictions (diagonal cells)')

In [ ]:
# ============================================================
# Full Classification Report — Random Forest
# ============================================================
print('=== Random Forest Classification Report ===')
print(classification_report(y_test, rfc_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# ============================================================
# Feature Importance — Top 20 Most Predictive Symptoms
# ============================================================
feat_importances = pd.Series(rfc.feature_importances_, index=X.columns)
top_features = feat_importances.nlargest(20)

fig, ax = plt.subplots(figsize=(12, 7))
palette = sns.color_palette('Purples_r', len(top_features))
bars = ax.barh(range(len(top_features)), top_features.values, color=palette)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels([s.replace('_', ' ').title() for s in top_features.index], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gini Impurity Reduction)', fontsize=11)
ax.set_title('Top 20 Most Predictive Symptoms (Random Forest)', fontsize=14, fontweight='bold', pad=15)

for bar, val in zip(bars, top_features.values):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
# Section 7: Save Models to Disk

In [ ]:
# ============================================================
# Save All Trained Models (for Flask web app)
# ============================================================

# Primary model (RandomForest — highest accuracy)
model = rfc   # alias used by the Flask app

model_files = {
    'svc_model.pkl'      : results['SVC']['model'],
    'rf_model.pkl'       : rfc,
    'gb_model.pkl'       : results['GradientBoosting']['model'],
    'knn_model.pkl'      : results['KNeighbors']['model'],
    'nb_model.pkl'       : results['MultinomialNB']['model'],
    'label_encoder.pkl'  : le,
}

for fname, obj in model_files.items():
    fpath = os.path.join(MODELS_DIR, fname)
    with open(fpath, 'wb') as f:
        pickle.dump(obj, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'✅ Saved: {fname:<30s}  {size_kb:>8.1f} KB')

print()
print('Primary model for Flask app: rf_model.pkl (Random Forest)')

In [ ]:
# ============================================================
# Verify: Reload & Test Saved Model
# ============================================================
model = pickle.load(open(os.path.join(MODELS_DIR, 'rf_model.pkl'), 'rb'))
le_loaded = pickle.load(open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb'))

# Test on sample
test_pred = model.predict(X_test.iloc[[0]].values)
print(f'Reload test — Predicted: {le_loaded.inverse_transform(test_pred)[0]}')
print(f'Reload test — Actual   : {le_loaded.inverse_transform([y_test[0]])[0]}')
print('\n✅ Model saved and reloaded successfully!')

---
# Section 8: Recommendation System Setup

Load all supplementary datasets and define helper functions used by the recommendation engine.

In [ ]:
# ============================================================
# Load Supplementary Recommendation Datasets
# ============================================================
sym_des     = pd.read_csv(os.path.join(DATASET_DIR, 'symtoms_df.csv'))
precautions = pd.read_csv(os.path.join(DATASET_DIR, 'precautions_df.csv'))
workout     = pd.read_csv(os.path.join(DATASET_DIR, 'workout_df.csv'))
description = pd.read_csv(os.path.join(DATASET_DIR, 'description.csv'))
medications = pd.read_csv(os.path.join(DATASET_DIR, 'medications.csv'))
diets       = pd.read_csv(os.path.join(DATASET_DIR, 'diets.csv'))

print('Supplementary datasets loaded:')
print(f'  sym_des     : {sym_des.shape}')
print(f'  precautions : {precautions.shape}')
print(f'  workout     : {workout.shape}')
print(f'  description : {description.shape}')
print(f'  medications : {medications.shape}')
print(f'  diets       : {diets.shape}')
print()
print('Medications sample:')
medications.head()

In [ ]:
# ============================================================
# Build Symptom Dictionary & Disease Mapping
# ============================================================

# Map symptom column name → column index (used for vectorising user input)
symptoms_dict = {'itching': 0, 'skin_rash': 1, 'nodal_skin_eruptions': 2, 'continuous_sneezing': 3,
                 'shivering': 4, 'chills': 5, 'joint_pain': 6, 'stomach_pain': 7, 'acidity': 8,
                 'ulcers_on_tongue': 9, 'muscle_wasting': 10, 'vomiting': 11, 'burning_micturition': 12,
                 'spotting_ urination': 13, 'fatigue': 14, 'weight_gain': 15, 'anxiety': 16,
                 'cold_hands_and_feets': 17, 'mood_swings': 18, 'weight_loss': 19, 'restlessness': 20,
                 'lethargy': 21, 'patches_in_throat': 22, 'irregular_sugar_level': 23, 'cough': 24,
                 'high_fever': 25, 'sunken_eyes': 26, 'breathlessness': 27, 'sweating': 28,
                 'dehydration': 29, 'indigestion': 30, 'headache': 31, 'yellowish_skin': 32,
                 'dark_urine': 33, 'nausea': 34, 'loss_of_appetite': 35, 'pain_behind_the_eyes': 36,
                 'back_pain': 37, 'constipation': 38, 'abdominal_pain': 39, 'diarrhoea': 40,
                 'mild_fever': 41, 'yellow_urine': 42, 'yellowing_of_eyes': 43, 'acute_liver_failure': 44,
                 'fluid_overload': 45, 'swelling_of_stomach': 46, 'swelled_lymph_nodes': 47,
                 'malaise': 48, 'blurred_and_distorted_vision': 49, 'phlegm': 50,
                 'throat_irritation': 51, 'redness_of_eyes': 52, 'sinus_pressure': 53,
                 'runny_nose': 54, 'congestion': 55, 'chest_pain': 56, 'weakness_in_limbs': 57,
                 'fast_heart_rate': 58, 'pain_during_bowel_movements': 59, 'pain_in_anal_region': 60,
                 'bloody_stool': 61, 'irritation_in_anus': 62, 'neck_pain': 63, 'dizziness': 64,
                 'cramps': 65, 'bruising': 66, 'obesity': 67, 'swollen_legs': 68,
                 'swollen_blood_vessels': 69, 'puffy_face_and_eyes': 70, 'enlarged_thyroid': 71,
                 'brittle_nails': 72, 'swollen_extremeties': 73, 'excessive_hunger': 74,
                 'extra_marital_contacts': 75, 'drying_and_tingling_lips': 76, 'slurred_speech': 77,
                 'knee_pain': 78, 'hip_joint_pain': 79, 'muscle_weakness': 80, 'stiff_neck': 81,
                 'swelling_joints': 82, 'movement_stiffness': 83, 'spinning_movements': 84,
                 'loss_of_balance': 85, 'unsteadiness': 86, 'weakness_of_one_body_side': 87,
                 'loss_of_smell': 88, 'bladder_discomfort': 89, 'foul_smell_of urine': 90,
                 'continuous_feel_of_urine': 91, 'passage_of_gases': 92, 'internal_itching': 93,
                 'toxic_look_(typhos)': 94, 'depression': 95, 'irritability': 96, 'muscle_pain': 97,
                 'altered_sensorium': 98, 'red_spots_over_body': 99, 'belly_pain': 100,
                 'abnormal_menstruation': 101, 'dischromic _patches': 102, 'watering_from_eyes': 103,
                 'increased_appetite': 104, 'polyuria': 105, 'family_history': 106,
                 'mucoid_sputum': 107, 'rusty_sputum': 108, 'lack_of_concentration': 109,
                 'visual_disturbances': 110, 'receiving_blood_transfusion': 111,
                 'receiving_unsterile_injections': 112, 'coma': 113, 'stomach_bleeding': 114,
                 'distention_of_abdomen': 115, 'history_of_alcohol_consumption': 116,
                 'fluid_overload.1': 117, 'blood_in_sputum': 118, 'prominent_veins_on_calf': 119,
                 'palpitations': 120, 'painful_walking': 121, 'pus_filled_pimples': 122,
                 'blackheads': 123, 'scurring': 124, 'skin_peeling': 125, 'silver_like_dusting': 126,
                 'small_dents_in_nails': 127, 'inflammatory_nails': 128, 'blister': 129,
                 'red_sore_around_nose': 130, 'yellow_crust_ooze': 131}

print(f'Total symptoms in dictionary : {len(symptoms_dict)}')
print(f'Total diseases in label enc  : {len(diseases_list)}')

In [ ]:
# ============================================================
# Helper Function: Fetch All Recommendations for a Disease
# ============================================================
def helper(dis):
    '''Return description, precautions, medications, diet, and workout for a given disease.'''
    # Disease description
    desc = description[description['Disease'] == dis]['Description']
    desc = ' '.join([w for w in desc])

    # Precautions (4 columns)
    pre = precautions[precautions['Disease'] == dis][
        ['Precaution_1', 'Precaution_2', 'Precaution_3', 'Precaution_4']
    ]
    pre = [col for col in pre.values]

    # Medications
    med = medications[medications['Disease'] == dis]['Medication']
    med = [m for m in med.values]

    # Diet
    die = diets[diets['Disease'] == dis]['Diet']
    die = [d for d in die.values]

    # Workout
    wrkout = workout[workout['disease'] == dis]['workout']

    return desc, pre, med, die, wrkout


# ============================================================
# Prediction Function: Symptoms → Disease Name
# ============================================================
def get_predicted_value(patient_symptoms):
    '''Predict disease name from a list of symptom strings.'''
    input_vector = np.zeros(len(X.columns))

    for symptom in patient_symptoms:
        symptom_clean = symptom.strip().lower().replace(' ', '_')
        if symptom_clean in X.columns:
            index = list(X.columns).index(symptom_clean)
            input_vector[index] = 1
        elif symptom_clean in symptoms_dict:
            input_vector[symptoms_dict[symptom_clean]] = 1

    prediction = model.predict([input_vector])
    return le.inverse_transform(prediction)[0]


print('✅ Helper functions defined!')
print('   - helper(disease)                 → returns desc, pre, med, die, wrkout')
print('   - get_predicted_value(symptoms[]) → returns predicted disease name')

---
# Section 9: Content-Based Medicine Filtering (TF-IDF + Cosine Similarity)

In [ ]:
# ============================================================
# Load or Build Medicine Database
# ============================================================
medicine_path = os.path.join(DATASET_DIR, 'medicine.csv')

if os.path.exists(medicine_path):
    medicine_df = pd.read_csv(medicine_path)
    print(f'✅ medicine.csv loaded — {len(medicine_df)} entries')
else:
    # Build from medications dataset
    med_data = []
    for _, row in medications.iterrows():
        disease  = row['Disease']
        meds_str = row['Medication']
        try:
            meds_list = eval(meds_str) if isinstance(meds_str, str) else [meds_str]
            for m in meds_list:
                med_data.append({
                    'Drug_Name'  : m.strip(),
                    'Reason'     : disease,
                    'Description': f'Recommended for {disease}'
                })
        except Exception:
            med_data.append({
                'Drug_Name'  : str(meds_str),
                'Reason'     : disease,
                'Description': f'Recommended for {disease}'
            })
    medicine_df = pd.DataFrame(med_data).drop_duplicates().reset_index(drop=True)
    medicine_df.to_csv(medicine_path, index=False)
    print(f'✅ medicine.csv created — {len(medicine_df)} entries')

medicine_df.head(10)

In [ ]:
# ============================================================
# TF-IDF Vectorization + Cosine Similarity Matrix
# ============================================================

# Create feature tags from Description + Reason
medicine_df['tags'] = (
    medicine_df['Description'].fillna('') + ' ' +
    medicine_df['Reason'].fillna('')
).str.lower()

# TF-IDF
vectorizer  = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(medicine_df['tags'])

print(f'TF-IDF matrix shape  : {tfidf_matrix.shape}')
print(f'Vocabulary size      : {len(vectorizer.vocabulary_)}')

# Cosine Similarity
similarity = cosine_similarity(tfidf_matrix)
print(f'Similarity matrix    : {similarity.shape}')
print(f'\nSample similarity score [0,1]: {similarity[0][1]:.4f}')

In [ ]:
# ============================================================
# Content-Based Medicine Recommendation Function
# ============================================================
def recommend_medicines(drug_name, n=5):
    '''Return top-N similar medicines using TF-IDF cosine similarity.'''
    # Exact match first
    matches = medicine_df[medicine_df['Drug_Name'].str.lower() == drug_name.lower()]
    if len(matches) == 0:
        # Partial match
        matches = medicine_df[medicine_df['Drug_Name'].str.lower().str.contains(drug_name.lower())]
    if len(matches) == 0:
        return [f"Drug '{drug_name}' not found in database."]

    idx       = matches.index[0]
    distances = sorted(enumerate(similarity[idx]), reverse=True, key=lambda x: x[1])

    print(f"Top {n} medicines similar to '{medicine_df.iloc[idx]['Drug_Name']}' "
          f"(Reason: {medicine_df.iloc[idx]['Reason']}):\n")
    recs = []
    for j, score in distances[1: n+1]:
        drug = medicine_df.iloc[j]
        print(f"  {len(recs)+1}. {drug['Drug_Name']:<35s} → {drug['Reason']:<30s} (score: {score:.4f})")
        recs.append(drug['Drug_Name'])
    return recs


# Save similarity matrix
with open(os.path.join(MODELS_DIR, 'similarity.pkl'), 'wb') as f:
    pickle.dump(similarity, f)
medicine_df.to_csv(os.path.join(DATASET_DIR, 'medicine.csv'), index=False)
print('✅ Similarity matrix saved to models/similarity.pkl')
print('✅ Medicine database saved to dataset/medicine.csv')

In [ ]:
# Quick test — recommend medicines similar to first entry
sample_drug = medicine_df.iloc[0]['Drug_Name']
print(f'Testing medicine recommendation for: "{sample_drug}"\n')
recs = recommend_medicines(sample_drug, n=5)
print('\n✅ Content-based medicine recommendation system working!')

---
# Section 10: End-to-End Testing

Test the full pipeline: symptoms → disease → description → precautions → medications → diet → workout

In [ ]:
# ============================================================
# Test Case 1: Migraine
# ============================================================
symptoms_input = 'headache,blurred_and_distorted_vision,visual_disturbances,stiff_neck,excessive_hunger'
user_symptoms  = [s.strip() for s in symptoms_input.split(',')]
user_symptoms  = [symptom.strip("[]' ") for symptom in user_symptoms]

predicted_disease = get_predicted_value(user_symptoms)
desc, pre, med, die, wrkout = helper(predicted_disease)

print('=' * 55)
print('🔬  SYMPTOM ANALYSIS')
print('=' * 55)
print(f'Symptoms entered : {symptoms_input}')
print()

print('=' * 55)
print('🏥  PREDICTED DISEASE')
print('=' * 55)
print(f'  {predicted_disease}')
print()

print('=' * 55)
print('📋  DESCRIPTION')
print('=' * 55)
print(f'  {desc}')
print()

print('=' * 55)
print('⚠️   PRECAUTIONS')
print('=' * 55)
i = 1
for p_i in pre[0] if pre else []:
    print(f'  {i}. {p_i}')
    i += 1
print()

print('=' * 55)
print('💊  MEDICATIONS')
print('=' * 55)
for m_i in med:
    print(f'  {i}. {m_i}')
    i += 1
print()

print('=' * 55)
print('🏋️   WORKOUT RECOMMENDATIONS')
print('=' * 55)
for w_i in wrkout:
    print(f'  {i}. {w_i}')
    i += 1
print()

print('=' * 55)
print('🥗  DIETS')
print('=' * 55)
for d_i in die:
    print(f'  {i}. {d_i}')
    i += 1

In [ ]:
# ============================================================
# Test Case 2: Impetigo (matches reference notebook test)
# ============================================================
symptoms_input = 'yellow_crust_ooze,red_sore_around_nose,small_dents_in_nails,inflammatory_nails,blister'
user_symptoms  = [s.strip() for s in symptoms_input.split(',')]
user_symptoms  = [symptom.strip("[]' ") for symptom in user_symptoms]

predicted_disease = get_predicted_value(user_symptoms)
desc, pre, med, die, wrkout = helper(predicted_disease)

print('=' * 55)
print('🔬  SYMPTOM ANALYSIS')
print('=' * 55)
print(f'Symptoms : {symptoms_input}')
print(f'\n🏥 Predicted Disease: {predicted_disease}')
print(f'\n📋 Description: {desc}')
print()

print('⚠️  Precautions:')
for i, p in enumerate(pre[0] if pre else [], 1):
    print(f'  {i}. {p}')

print()
print('💊 Medications:')
for i, m in enumerate(med, 1):
    print(f'  {i}. {m}')

print()
print('🏋️  Workout:')
for i, w in enumerate(wrkout, 1):
    print(f'  {i}. {w}')

print()
print('🥗 Diet:')
for i, d in enumerate(die, 1):
    print(f'  {i}. {d}')

---
# Section 11: Multiple Test Cases (Batch Validation)

In [ ]:
# ============================================================
# Batch Symptom Test Sets — from reference implementation
# ============================================================
test_cases = [
    ('Migraine',              'headache,blurred_and_distorted_vision,visual_disturbances,stiff_neck,excessive_hunger'),
    ('Allergic Rhinitis',     'continuous_sneezing,runny_nose,congestion,throat_irritation,sinus_pressure'),
    ('Hypertension',          'headache,chest_pain,dizziness,loss_of_balance,lack_of_concentration'),
    ('Food Poisoning',        'vomiting,diarrhoea,abdominal_pain,dehydration,nausea'),
    ('Dengue',                'high_fever,joint_pain,muscle_pain,headache,fatigue'),
    ('Gastritis',             'acidity,indigestion,abdominal_pain,nausea,loss_of_appetite'),
    ('Bronchitis',            'cough,breathlessness,chest_pain,fatigue,phlegm'),
    ('Urinary Tract Infection','burning_micturition,spotting_ urination,abdominal_pain,nausea'),
    ('Hypothyroidism',        'weight_gain,fatigue,depression,lethargy,dizziness'),
    ('Anxiety Disorder',      'restlessness,sweating,fatigue,dizziness,headache'),
    ('Heat Stroke',           'high_fever,headache,sweating,dizziness,fatigue'),
    ('Conjunctivitis',        'redness_of_eyes,itching,skin_rash,headache,watering_from_eyes'),
    ('Allergic Rhinitis',     'continuous_sneezing,runny_nose,congestion,headache,fatigue'),
    ('Impetigo',              'yellow_crust_ooze,red_sore_around_nose,small_dents_in_nails,inflammatory_nails,blister'),
]

print('\n=========== MODEL TEST RESULTS ===========\n')
correct = 0
for i, (expected, case) in enumerate(test_cases, 1):
    symptoms   = [s.strip() for s in case.split(',')]
    prediction = get_predicted_value(symptoms)
    match      = '✅' if expected.lower() in prediction.lower() else '❌'
    if expected.lower() in prediction.lower():
        correct += 1
    print(f'Test {i:2d} {match}  |  Expected: {expected:<30s}  |  Predicted: {prediction}')
    print(f'         Symptoms: {case}')
    print('--------------------------------------')

print(f'\nBatch Accuracy: {correct}/{len(test_cases)} = {correct/len(test_cases)*100:.1f}%')

---
# Section 12: Summary & Next Steps

## 🎯 What Was Built

| Component | Algorithm | Status |
|-----------|-----------|--------|
| Disease Prediction | Random Forest (200 trees, depth=15) | ✅ Saved |
| Disease Prediction | SVC (RBF kernel) | ✅ Saved |
| Disease Prediction | Gradient Boosting (150 trees, lr=0.05) | ✅ Saved |
| Disease Prediction | KNeighbors (k=7, distance) | ✅ Saved |
| Disease Prediction | Multinomial Naive Bayes | ✅ Saved |
| Medicine Recommendation | TF-IDF + Cosine Similarity | ✅ Saved |
| Label Encoder | sklearn LabelEncoder | ✅ Saved |

## 📦 Files Generated

| File | Purpose |
|------|---------|
| `models/rf_model.pkl` | Primary disease prediction model (Flask app) |
| `models/svc_model.pkl` | SVC disease prediction model |
| `models/gb_model.pkl` | Gradient Boosting model |
| `models/knn_model.pkl` | K-Nearest Neighbors model |
| `models/nb_model.pkl` | Naive Bayes model |
| `models/label_encoder.pkl` | Disease label encoder |
| `models/similarity.pkl` | Medicine cosine similarity matrix |
| `dataset/medicine.csv` | Drug database for content-based filtering |

## 🚀 Next Steps
1. **Run Flask App**: `python app.py` from the `HealthAI/` directory
2. **Open browser**: Navigate to `http://localhost:5000`
3. **Enter symptoms** in the UI to get disease predictions and personalized recommendations
4. **Extend**: Add collaborative filtering (SVD/matrix factorization), patient history, real-time lab integration

In [ ]:
# ============================================================
# Final Verification — List All Saved Artifacts
# ============================================================
print('=' * 60)
print('📦 SAVED MODEL FILES:')
print('=' * 60)
for f in sorted(os.listdir(MODELS_DIR)):
    fpath   = os.path.join(MODELS_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<35s}  {size_kb:>8.1f} KB')

print()
print('=' * 60)
print('📁 DATASET FILES:')
print('=' * 60)
for f in sorted(os.listdir(DATASET_DIR)):
    fpath   = os.path.join(DATASET_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<35s}  {size_kb:>8.1f} KB')

print()
print('=' * 60)
print('🏥 HealthAI Notebook Complete!')
print('   Run: python app.py  → Open: http://localhost:5000')
print('=' * 60)